# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/FatimaNdeem/Flyrank-ml-internship./blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [9]:
!git clone "https://github.com/FatimaNdeem/Flyrank-ml-internship." /content/Flyrank-ml-internship.

fatal: destination path '/content/Flyrank-ml-internship.' already exists and is not an empty directory.


In [10]:
%cd /content/Flyrank-ml-internship.
!ls -lh data/raw/

/content/Flyrank-ml-internship.
total 6.5M
-rw-r--r-- 1 root root 6.5M Aug 21 22:06 content_refresh_anonymized.csv


In [11]:
import pandas as pd
import numpy as np

# Make sure we are inside the repository
%cd /content/Flyrank-ml-internship.

# Load the dataset
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print("Dataset shape:", df.shape)
print("Number of columns:", len(df.columns))

/content/Flyrank-ml-internship.
Dataset shape: (30000, 44)
Number of columns: 44


## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

#Finding 1 — Freshness and content depth

The paper reports that freshness is associated with stronger content performance, but the effect changes when content depth is considered. For example, among pages with 3,500+ words, content updated within 0–30 days had an average health score of 28.7 compared with 15.9 for pages 181+ days old. The paper therefore presents freshness as something that amplifies existing quality rather than as a guarantee that updating any page will improve it. The label in this finding is the paper's health score, which is a FlyRank composite rather than a Google metric. The methodology states that health score combines impressions, position, CTR, and scroll depth.The validation design supports this as an observational comparison between content groups, but it does not establish that refreshing a page causes the health score to increase. The paper itself says that its study is observational and that correlations do not prove causation.



## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

**Before:** My initial modeling result used the same starter dataset and produced a Random Forest F1 score of 0.6096 on the grouped test set. Precision was 0.5766 and recall was 0.6466.

**After:** I re-evaluated the model using the same client-grouped 80/20 split, keeping entire clients out of either the training or test set. The split contained 23,837 training rows and 6,163 test rows, with 25 training clients and 7 test clients and zero client overlap. The Random Forest achieved 0.5768 accuracy, 0.5766 precision, 0.6466 recall, and 0.6096 F1.

Because the submitted Week-5 model already used an honest grouped split, the validation audit did not reveal a performance change from changing the split. This is useful evidence that the reported Week-5 result was already based on client holdout rather than a random row split.

The result should still be treated as directional because the model is evaluated on a 30,000-row anonymized starter slice rather than the full warehouse.**bold text**

In [12]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Recreate the same target
df["is_declining"] = (
    df["impressions_last_30d"]
    < 0.8 * df["impressions_prev_30d"]
).astype(int)

# Same honest grouped split used in ML-08
groups = df["client_id"]

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(
        df,
        df["is_declining"],
        groups=groups
    )
)

print("Train rows:", len(train_idx))
print("Test rows:", len(test_idx))

train_clients = set(df.iloc[train_idx]["client_id"])
test_clients = set(df.iloc[test_idx]["client_id"])

print("Train clients:", len(train_clients))
print("Test clients:", len(test_clients))
print("Client overlap:", len(train_clients & test_clients))

print("\nML-08 submitted Random Forest metrics:")
print("Accuracy: 0.5768")
print("Precision: 0.5766")
print("Recall: 0.6466")
print("F1: 0.6096")

Train rows: 23837
Test rows: 6163
Train clients: 25
Test clients: 7
Client overlap: 0

ML-08 submitted Random Forest metrics:
Accuracy: 0.5768
Precision: 0.5766
Recall: 0.6466
F1: 0.6096


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

I audited the final feature set used in the submitted Week-5 model. The two variables directly used to construct the is_declining target — impressions_last_30d and impressions_prev_30d — were excluded from the final Random Forest feature list.

I also checked for future-looking variables such as future, next_30d, next_60d, and next_90d. None were included in the final feature set.

Finally, the train/test split was grouped by client_id, and the resulting split had zero client overlap.

The audit therefore found no direct target-definition features, no explicitly future-looking features, and no client overlap in the submitted model.

In [15]:
# Feature groups
numeric_features = [
    "imp_prev30",
    "visible_queries",
    "rare_share",
    "anon_share",
    "top_query_share"
]

categorical_features = []

# Features used to define the target
target_definition_features = [
    "impressions_last_30d",
    "impressions_prev_30d"
]

# Terms that indicate potential future information
future_terms = [
    "future",
    "next_30d",
    "next_60d",
    "next_90d"
]

# Check for target-definition features being used as predictors
used_target_features = [
    col for col in numeric_features + categorical_features
    if col in target_definition_features
]

# Check for future-looking features
future_like_features = [
    col for col in numeric_features + categorical_features
    if any(term in col.lower() for term in future_terms)
]

# Check for client leakage between train and test
train_clients = set(df.iloc[train_idx]["client_id"])
test_clients = set(df.iloc[test_idx]["client_id"])

client_overlap = len(train_clients & test_clients)

print("Target-definition features used:", used_target_features)
print("Future-like features used:", future_like_features)
print("Client overlap:", client_overlap)

leakage_free = (
    len(used_target_features) == 0
    and len(future_like_features) == 0
    and client_overlap == 0
)

print("\nLeakage audit passed:", leakage_free)

Target-definition features used: []
Future-like features used: []
Client overlap: 0

Leakage audit passed: True


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

**Original claim:**
"The Random Forest can identify declining content and is better than the baseline."

**Now claim:**
"On the 30,000-row anonymized starter dataset, the Random Forest achieved a measured F1 score of 0.6096 on the client-grouped test set, compared with 0.4629 for the Week-4 baseline. The model also achieved higher recall but lower precision than the baseline. These results are directional evidence that the observed feature set can support decline-risk classification on this sample; they do not prove that the model will generalize to the full warehouse or that a flagged page will recover after being refreshed. The model should therefore be treated as decision-support rather than as proof of a required action."

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.